# Notebook 1 — A2UI through Strands

**IBS Agentic AI Practitioner Bootcamp · Final Day**

You already built agents that reason, call tools, talk to each other, and run on AWS. Every one of them returns text. This notebook adds the layer a human actually touches: the agent emits a structured **A2UI** payload, you gate the risky action, you log everything, and you watch it through an admin view.

Anchor use case: **TravelMind**, the airline support agent. PNR `JX48Q2`.

### What you build, step by step

| Step | Verb | What you add |
|---|---|---|
| 1 | - | An A2UI v0.8 payload by hand, then builders |
| 2 | SHOW + STRUCTURE | TravelMind flight options as a real surface |
| 3 | (safety) | A validator, so malformed UI degrades to text |
| 4 | STRUCTURE | A Strands `@tool` that emits and captures A2UI |
| 5 | - | A real Strands agent on Bedrock driving the tool |
| 6 | GATE | An approval flow with the tool call shown |
| 7 | RECORD | An audit log of every payload and user action |
| 8 | SHOW | A Streamlit admin view over the run |
| 9 | - | Dos and don'ts |

Every cell runs top to bottom. Live Bedrock calls are off by default, so the build is reproducible in class. You flip one flag to go live.

## 0. Setup and config

### Install (run once)

```bash
python -m venv .venv && source .venv/bin/activate
pip install strands-agents bedrock-agentcore streamlit
```

### Config facts that bite if you skip them

| Setting | Value | Why it matters |
|---|---|---|
| Region | `us-east-1` | TravelMind's account and model access live here |
| Model | `us.anthropic.claude-haiku-4-5-20251001-v1:0` | This is an **inference profile** id. A bare model id throws `ValidationException` |
| IAM action | `bedrock:InvokeModel` | Strands uses the Converse API under the hood, and `InvokeModel` covers it. There is no `bedrock:Converse` action |
| Model access | Enabled in Bedrock console | Console → Model access → enable Claude Haiku 4.5 (and Sonnet 4 for the swap) |
| Credentials | env vars or `aws configure` | Only needed when you set `run_live_agent = True` |

In [2]:
# Commented so this notebook never reinstalls during class. Run the pip line once in your venv.
# !pip install strands-agents bedrock-agentcore streamlit

import json
from datetime import datetime, timezone

import strands
from strands import Agent, tool
from strands.models import BedrockModel

CONFIG = {
    "aws_region": "us-east-1",
    "model_id": "us.anthropic.claude-haiku-4-5-20251001-v1:0",   # inference profile (mandatory)
    "model_swap": "us.anthropic.claude-sonnet-4-20250514-v1:0",  # Sonnet 4 cross-region profile
    "pnr": "JX48Q2",
    "run_live_agent": True,   # True -> calls Bedrock; needs creds + model access in us-east-1
}

print("strands-agents:", getattr(strands, "__version__", "see pip show strands-agents"))
print("config:", json.dumps(CONFIG, indent=2))

strands-agents: see pip show strands-agents
config: {
  "aws_region": "us-east-1",
  "model_id": "us.anthropic.claude-haiku-4-5-20251001-v1:0",
  "model_swap": "us.anthropic.claude-sonnet-4-20250514-v1:0",
  "pnr": "JX48Q2",
  "run_live_agent": true
}


## 1. An A2UI payload, by hand

A2UI v0.8 facts, verified against a2ui.org:

| Idea | Shape |
|---|---|
| Message types | `surfaceUpdate`, `dataModelUpdate`, `beginRendering`, `deleteSurface` |
| Component | `{ "id": "...", "component": { "<Type>": { ...props } } }` (wraps exactly one type) |
| Bound value | `{ "literalString": "..." }` or `{ "path": "/x" }` or both |
| Container children | `{ "children": { "explicitList": ["id1", "id2"] } }` |
| Button | `{ "Button": { "child": "label-id", "action": { "name": "..." } } }` |

The component list is a flat **adjacency list**: parents point at children by string id, and the client rebuilds the tree at render time. Flat is easier for an LLM to generate than a deep nested tree.

Here is the raw wire shape for a one-line "Hello" surface.

In [3]:
hello_by_hand = [
    {"surfaceUpdate": {"surfaceId": "demo", "components": [
        {"id": "root",  "component": {"Column": {"children": {"explicitList": ["greeting"]}}}},
        {"id": "greeting", "component": {"Text": {"text": {"literalString": "Hello from A2UI"}}}},
    ]}},
    {"beginRendering": {"surfaceId": "demo", "root": "root"}},
]
print(json.dumps(hello_by_hand, indent=2))

[
  {
    "surfaceUpdate": {
      "surfaceId": "demo",
      "components": [
        {
          "id": "root",
          "component": {
            "Column": {
              "children": {
                "explicitList": [
                  "greeting"
                ]
              }
            }
          }
        },
        {
          "id": "greeting",
          "component": {
            "Text": {
              "text": {
                "literalString": "Hello from A2UI"
              }
            }
          }
        }
      ]
    }
  },
  {
    "beginRendering": {
      "surfaceId": "demo",
      "root": "root"
    }
  }
]


Hand-writing that JSON for every screen is a mistake. Builders make the structure obvious and keep ids consistent.

In [4]:
V08_CATALOG_ID = "https://a2ui.org/specification/v0_8/standard_catalog_definition.json"
# Column / Row / Text / Button are standard v0.8 primitives.
# MetricCard is a CUSTOM component the client registers (the Strands reference app ships custom ones).
KNOWN_TYPES = {"Column", "Row", "Text", "Button", "MetricCard"}
CONTAINER_LIST_TYPES = {"Column", "Row"}


def now_iso():
    return datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")


# ---- component builders (each returns one adjacency-list node) ----
def text(cid, value=None, path=None):
    bound = {}
    if path is not None:
        bound["path"] = path
    if value is not None:
        bound["literalString"] = value
    return {"id": cid, "component": {"Text": {"text": bound}}}


def button(cid, child_id, action_name, context=None):
    action = {"name": action_name}
    if context:
        action["context"] = context  # wire spec: context is a list of data-model paths to extract
    return {"id": cid, "component": {"Button": {"child": child_id, "action": action}}}


def column(cid, child_ids):
    return {"id": cid, "component": {"Column": {"children": {"explicitList": list(child_ids)}}}}


def row(cid, child_ids):
    return {"id": cid, "component": {"Row": {"children": {"explicitList": list(child_ids)}}}}


def metric_card(cid, label, value):
    return {"id": cid, "component": {"MetricCard": {"label": {"literalString": label},
                                                    "value": {"literalString": value}}}}


# ---- message builders ----
def surface_update(surface_id, components):
    return {"surfaceUpdate": {"surfaceId": surface_id, "components": components}}


def data_model_update(surface_id, contents):
    return {"dataModelUpdate": {"surfaceId": surface_id, "contents": contents}}


def begin_rendering(surface_id, root_id):
    return {"beginRendering": {"surfaceId": surface_id, "root": root_id}}


def delete_surface(surface_id):
    return {"deleteSurface": {"surfaceId": surface_id}}


def user_action(name, surface_id, source_component_id, context=None):
    return {"userAction": {"name": name, "surfaceId": surface_id,
                           "sourceComponentId": source_component_id,
                           "timestamp": now_iso(), "context": context or {}}}


# rebuild Hello with builders
hello = [surface_update("demo", [column("root", ["greeting"]),
                                 text("greeting", "Hello from A2UI")]),
         begin_rendering("demo", "root")]
print("same payload, fewer keystrokes:")
print(json.dumps(hello, indent=2))

same payload, fewer keystrokes:
[
  {
    "surfaceUpdate": {
      "surfaceId": "demo",
      "components": [
        {
          "id": "root",
          "component": {
            "Column": {
              "children": {
                "explicitList": [
                  "greeting"
                ]
              }
            }
          }
        },
        {
          "id": "greeting",
          "component": {
            "Text": {
              "text": {
                "literalString": "Hello from A2UI"
              }
            }
          }
        }
      ]
    }
  },
  {
    "beginRendering": {
      "surfaceId": "demo",
      "root": "root"
    }
  }
]


We need to *see* surfaces in this notebook without a browser. This tiny renderer walks the real payload, resolves data-model paths, and prints a text picture. It is our own renderer over the genuine A2UI structure, not a mock.

In [5]:
def _index(messages):
    comps, root, dm = {}, None, {}
    for msg in messages:
        if "surfaceUpdate" in msg:
            for c in msg["surfaceUpdate"]["components"]:
                comps[c["id"]] = c["component"]
        elif "beginRendering" in msg:
            root = msg["beginRendering"]["root"]
        elif "dataModelUpdate" in msg:
            for entry in msg["dataModelUpdate"]["contents"]:
                if "valueString" in entry:
                    dm["/" + entry["key"]] = entry["valueString"]
    return comps, root, dm


def _resolve(bound, dm):
    if not isinstance(bound, dict):
        return str(bound)
    if "literalString" in bound and "path" in bound:
        return dm.get(bound["path"], bound["literalString"])
    if "path" in bound:
        return dm.get(bound["path"], f"<{bound['path']}>")
    return bound.get("literalString", "")


def render_ascii(messages, depth_marker="  "):
    comps, root, dm = _index(messages)
    out = []

    def walk(cid, depth):
        node = comps.get(cid)
        if node is None:
            out.append(depth_marker * depth + f"[missing:{cid}]"); return
        ctype = list(node.keys())[0]
        body = node[ctype]
        pad = depth_marker * depth
        if ctype == "Text":
            out.append(pad + _resolve(body.get("text", {}), dm))
        elif ctype == "Button":
            child = body.get("child")
            label = _resolve(comps.get(child, {}).get("Text", {}).get("text", {}), dm) if child in comps else child
            out.append(pad + f"[ {label} ]  -> action:{body['action']['name']}")
        elif ctype == "MetricCard":
            out.append(pad + f"+-- {_resolve(body['label'], dm)}: {_resolve(body['value'], dm)} --+")
        elif ctype in CONTAINER_LIST_TYPES:
            out.append(pad + f"{ctype}:")
            for ch in body.get("children", {}).get("explicitList", []):
                walk(ch, depth + 1)

    walk(root, 0) if root else out.append("(no beginRendering root set)")
    s = "\n".join(out)
    print(s)
    return s


print("rendered:")
render_ascii(hello)

rendered:
Column:
  Hello from A2UI


'Column:\n  Hello from A2UI'

## 2. TravelMind flight options — SHOW + STRUCTURE

The agent's native answer would be: *"I found 3 flights. JX490 18:05 9,400. JX511 13:40 11,800. JX482 09:15 14,200."* A human has to parse that. The structured version is scannable, comparable, and each row carries an action.

In [6]:
FLIGHTS = [
    {"id": "JX490", "depart": "18:05", "fare": "9,400"},
    {"id": "JX511", "depart": "13:40", "fare": "11,800"},
    {"id": "JX482", "depart": "09:15", "fare": "14,200"},
]


def build_options_surface(surface_id, flights):
    comps, row_ids = [], []
    for f in flights:
        t_flt, t_fare = f"flt-{f['id']}", f"fare-{f['id']}"
        b_lbl, b_btn = f"lbl-{f['id']}", f"book-{f['id']}"
        comps += [
            text(t_flt, f"{f['id']}  dep {f['depart']}"),
            text(t_fare, f"INR {f['fare']}"),
            text(b_lbl, "Rebook"),
            button(b_btn, b_lbl, "select_flight", context=[f"/flights/{f['id']}"]),
            row(f"row-{f['id']}", [t_flt, t_fare, b_btn]),
        ]
        row_ids.append(f"row-{f['id']}")
    comps += [text("hdr", "Available flights"), column("root", ["hdr"] + row_ids)]
    return [surface_update(surface_id, comps), begin_rendering(surface_id, "root")]


options = build_options_surface("flights", FLIGHTS)
print("what the user would see:\n")
render_ascii(options)

what the user would see:

Column:
  Available flights
  Row:
    JX490  dep 18:05
    INR 9,400
    [ Rebook ]  -> action:select_flight
  Row:
    JX511  dep 13:40
    INR 11,800
    [ Rebook ]  -> action:select_flight
  Row:
    JX482  dep 09:15
    INR 14,200
    [ Rebook ]  -> action:select_flight


'Column:\n  Available flights\n  Row:\n    JX490  dep 18:05\n    INR 9,400\n    [ Rebook ]  -> action:select_flight\n  Row:\n    JX511  dep 13:40\n    INR 11,800\n    [ Rebook ]  -> action:select_flight\n  Row:\n    JX482  dep 09:15\n    INR 14,200\n    [ Rebook ]  -> action:select_flight'

## 3. A validator — so bad UI never reaches the user

The agent is an LLM. It will sometimes emit malformed JSON, a dangling child reference, or a component type the client cannot render. The rule: schema-check the payload, and on failure fall back to plain text. Never leave the user with a blank surface.

The validator checks four things: exactly one message key, every component wraps exactly one known type, child references resolve, and the `beginRendering` root exists.

In [7]:
TOP_KEYS = {"surfaceUpdate", "dataModelUpdate", "beginRendering", "deleteSurface", "userAction"}


def validate(messages):
    """Return a list of error strings. Empty means structurally valid."""
    errors, comp_ids, roots_needed, child_refs = [], set(), [], []
    for i, msg in enumerate(messages):
        keys = [k for k in msg if k in TOP_KEYS]
        if len(keys) != 1:
            errors.append(f"msg[{i}] needs exactly one A2UI key, found {list(msg.keys())}"); continue
        kind = keys[0]
        if kind == "surfaceUpdate":
            for c in msg["surfaceUpdate"].get("components", []):
                if "id" not in c or "component" not in c:
                    errors.append(f"msg[{i}] component missing id/component: {c}"); continue
                comp_ids.add(c["id"])
                tks = list(c["component"].keys())
                if len(tks) != 1:
                    errors.append(f"msg[{i}] component '{c['id']}' must wrap one type, found {tks}"); continue
                ctype, body = tks[0], c["component"][tks[0]]
                if ctype not in KNOWN_TYPES:
                    errors.append(f"msg[{i}] component '{c['id']}' unknown type '{ctype}'")
                if ctype in CONTAINER_LIST_TYPES:
                    for ch in body.get("children", {}).get("explicitList", []):
                        child_refs.append((c["id"], ch))
                if ctype == "Button" and "child" in body:
                    child_refs.append((c["id"], body["child"]))
        elif kind == "beginRendering":
            roots_needed.append(msg["beginRendering"].get("root"))
    for parent, ref in child_refs:
        if ref not in comp_ids:
            errors.append(f"component '{parent}' references missing child '{ref}'")
    for r in roots_needed:
        if r not in comp_ids:
            errors.append(f"beginRendering root '{r}' is not defined")
    return errors


print("good payload ->", validate(options))

broken = [surface_update("x", [column("root", ["ghost-child"]),
                               {"id": "bad", "component": {"Text": {}, "Button": {}}}]),
          begin_rendering("x", "no-root")]
print("\nbroken payload ->")
for e in validate(broken):
    print("  -", e)

good payload -> []

broken payload ->
  - msg[0] component 'bad' must wrap one type, found ['Text', 'Button']
  - component 'root' references missing child 'ghost-child'
  - beginRendering root 'no-root' is not defined


In [8]:
def render_or_fallback(messages, fallback_text):
    """Render only if valid; otherwise return plain text. This is graceful degradation."""
    errs = validate(messages)
    if errs:
        print("VALIDATION FAILED, falling back to text:")
        print(" ", fallback_text)
        return {"mode": "text", "text": fallback_text, "errors": errs}
    render_ascii(messages)
    return {"mode": "surface", "messages": messages}

_ = render_or_fallback(broken, "I found 3 flights: JX490, JX511, JX482.")

VALIDATION FAILED, falling back to text:
  I found 3 flights: JX490, JX511, JX482.


## 4. A Strands tool that emits and captures A2UI

The pattern from the Strands reference architecture: A2UI is produced by a normal `@tool`, captured, and shipped. We capture into a small store so we can inspect every payload the agent emitted.

The tool returns a short text summary to the model (so the agent knows it succeeded) and records the full payload on the side.

In [9]:
class A2UICapture:
    def __init__(self):
        self.payloads = []

    def record(self, surface_id, messages, kind):
        self.payloads.append({"surfaceId": surface_id, "kind": kind,
                              "ts": now_iso(), "messages": messages})


CAPTURE = A2UICapture()


@tool
def render_flight_options(origin: str, destination: str) -> str:
    """Render flight options as a structured surface. Use when showing choices the user must compare."""
    msgs = build_options_surface("flights", FLIGHTS)
    if validate(msgs):
        return "Could not render options; tell the user the flights in plain text."
    CAPTURE.record("flights", msgs, "options")
    return f"Rendered {len(FLIGHTS)} flight options from {origin} to {destination} on surface 'flights'."


# direct call, no agent, no AWS
print(render_flight_options("BLR", "BOM"))
print("captured payloads so far:", len(CAPTURE.payloads))
print("\nlatest captured surface:")
render_ascii(CAPTURE.payloads[-1]["messages"])

Rendered 3 flight options from BLR to BOM on surface 'flights'.
captured payloads so far: 1

latest captured surface:
Column:
  Available flights
  Row:
    JX490  dep 18:05
    INR 9,400
    [ Rebook ]  -> action:select_flight
  Row:
    JX511  dep 13:40
    INR 11,800
    [ Rebook ]  -> action:select_flight
  Row:
    JX482  dep 09:15
    INR 14,200
    [ Rebook ]  -> action:select_flight


'Column:\n  Available flights\n  Row:\n    JX490  dep 18:05\n    INR 9,400\n    [ Rebook ]  -> action:select_flight\n  Row:\n    JX511  dep 13:40\n    INR 11,800\n    [ Rebook ]  -> action:select_flight\n  Row:\n    JX482  dep 09:15\n    INR 14,200\n    [ Rebook ]  -> action:select_flight'

## 5. A real Strands agent on Bedrock

Now the agent decides when to render. We build a `BedrockModel` on the inference profile and register the tool. A capture hook logs every tool the agent calls (the production way to intercept), using the `AfterToolCallEvent` lifecycle event.

`BedrockModel` constructs lazily, so building the agent does **not** call AWS. Set `run_live_agent = True` (and have creds) to let Claude drive. With it off, we call the tool directly to simulate the same outcome, so the rest of the notebook is deterministic.

In [ ]:
from strands.hooks import HookProvider, HookRegistry
try:
    from strands.hooks.events import AfterToolCallEvent          # strands 1.4x+
except ImportError:
    from strands.hooks.events import AfterToolInvocationEvent as AfterToolCallEvent  # older


class CaptureHook(HookProvider):
    """Logs every tool the agent invokes. The production way to intercept tool output."""
    def register_hooks(self, registry: HookRegistry) -> None:
        registry.add_callback(AfterToolCallEvent, self.after)

    def after(self, event) -> None:
        name = getattr(getattr(event, "selected_tool", None), "tool_name", None)
        if name:
            print(f"  [hook] agent called tool: {name}")


SYSTEM_PROMPT = (
    "You are TravelMind, an airline support agent. "
    "When the user asks to see or choose flights, call render_flight_options. "
    "Never dump flight lists as prose when a surface fits."
)

model = BedrockModel(model_id=CONFIG["model_id"], region_name=CONFIG["aws_region"])
agent = Agent(model=model, tools=[render_flight_options],
              hooks=[CaptureHook()], system_prompt=SYSTEM_PROMPT)

print("agent built (no AWS call yet)")
print("  model id:", agent.model.config.get("model_id"))
print("  tools   :", list(agent.tool_names))

if CONFIG["run_live_agent"]:
    print("\nLIVE: asking the agent to show flights...")
    result = agent("Show me flights from Bangalore to Mumbai for tomorrow - 10th June.")
    print("\nagent said:", str(result).strip()[:300])
else:
    print("\nrun_live_agent is False -> simulating the agent's tool call directly:")
    print(" ", render_flight_options("BLR", "BOM"))

agent built (no AWS call yet)
  model id: us.anthropic.claude-haiku-4-5-20251001-v1:0
  tools   : ['render_flight_options']

LIVE: asking the agent to show flights...
I'd be happy to help you see flights from Bangalore to Mumbai for tomorrow. However, I need to know today's date to determine which date is "tomorrow."

Could you please provide:
- Today's date (or the specific date you'd like to travel)?

Alternatively, if you can provide the specific travel date, I can show you the available flights right away.
agent said: I'd be happy to help you see flights from Bangalore to Mumbai for tomorrow. However, I need to know today's date to determine which date is "tomorrow."

Could you please provide:
- Today's date (or the specific date you'd like to travel)?

Alternatively, if you can provide the specific travel date, 


## 6. The approval flow — GATE

This is where UI stops being cosmetic and becomes a control. Two halves:

1. **Visibility:** show the tool call and its arguments before anything runs.
2. **The gate:** Approve or Reject buttons. The click returns a `userAction` the agent waits on.

The hard rule: **authorize on the server.** The button is convenience. The real decision happens in `authorize()`, which never trusts client-sent context blindly.

In [10]:
def build_approval_surface(surface_id, flight):
    comps = [
        text("q", f"Rebook {flight['id']} for INR {flight['fare']}?"),
        text("tool", f"tool: book_flight   flight={flight['id']}   pnr={CONFIG['pnr']}"),
        text("ok-l", "Approve"), button("ok", "ok-l", "approve_booking", context=[f"/pending/{flight['id']}"]),
        text("no-l", "Reject"),  button("no", "no-l", "reject_booking"),
        column("root", ["q", "tool", "ok", "no"]),
    ]
    return [surface_update(surface_id, comps), begin_rendering(surface_id, "root")]


def build_confirmation_surface(surface_id, flight):
    comps = [metric_card("m", "Booking confirmed", f"{flight['id']} . INR {flight['fare']}"),
             text("p", f"PNR {CONFIG['pnr']} updated"), column("root", ["m", "p"])]
    return [surface_update(surface_id, comps), begin_rendering(surface_id, "root")]


def authorize(action, context, role, amount=None):
    """Server-side gate. Returns (allowed, reason). Never trust the client to self-authorize."""
    if action == "book_flight":
        if role not in ("customer", "agent"):
            return False, f"role '{role}' may not book"
        if role == "customer" and amount and amount > 50000:
            return False, "amount over customer ceiling"
        return True, "permitted"
    return True, "no gate for this action"


chosen = FLIGHTS[1]   # JX511
approval = build_approval_surface("approve", chosen)
CAPTURE.record("approve", approval, "approval")
print("step 1 visibility + gate:\n")
render_ascii(approval)

step 1 visibility + gate:



Column:
  Rebook JX511 for INR 11,800?
  tool: book_flight   flight=JX511   pnr=JX48Q2
  [ Approve ]  -> action:approve_booking
  [ Reject ]  -> action:reject_booking


'Column:\n  Rebook JX511 for INR 11,800?\n  tool: book_flight   flight=JX511   pnr=JX48Q2\n  [ Approve ]  -> action:approve_booking\n  [ Reject ]  -> action:reject_booking'

In [11]:
# step 2: the user taps Approve -> a userAction comes back -> the server gate decides
incoming = user_action("approve_booking", "approve", "ok", {"flight": chosen["id"]})
fare_int = int(chosen["fare"].replace(",", ""))
allowed, reason = authorize("book_flight", incoming["userAction"]["context"], role="customer", amount=fare_int)
print("userAction :", incoming["userAction"]["name"], incoming["userAction"]["context"])
print("authorize  :", allowed, "-", reason)


@tool
def book_flight(flight_id: str) -> str:
    """Book a flight. Only call after an approved userAction has cleared the server gate."""
    f = next(x for x in FLIGHTS if x["id"] == flight_id)
    confirm = build_confirmation_surface("confirm", f)
    CAPTURE.record("confirm", confirm, "confirmation")
    return f"Booked {flight_id}; confirmation surface emitted."


if allowed:
    print("\n", book_flight(chosen["id"]))
    print()
    render_ascii(CAPTURE.payloads[-1]["messages"])
else:
    print("blocked, no booking performed")

userAction :

 approve_booking {'flight': 'JX511'}
authorize  : True - permitted

 Booked JX511; confirmation surface emitted.

Column:
  +-- Booking confirmed: JX511 . INR 11,800 --+
  PNR JX48Q2 updated


## 7. The audit log — RECORD

If the agent generated the UI, you must be able to reproduce exactly what the user saw and did. Log the payload emitted, the user action received, the authorization decision, and the tool that ran, each with a timestamp and a role.

In [12]:
AUDIT = []


def audit(kind, **fields):
    AUDIT.append({"ts": now_iso(), "kind": kind, **fields})


# rebuild the run, logging each step
AUDIT.clear()
CAPTURE.payloads.clear()

opts = build_options_surface("flights", FLIGHTS)
CAPTURE.record("flights", opts, "options")
audit("ui_emitted", surface="flights", role="customer", components=len(opts[0]["surfaceUpdate"]["components"]))

appr = build_approval_surface("approve", chosen)
CAPTURE.record("approve", appr, "approval")
audit("ui_emitted", surface="approve", role="customer", components=len(appr[0]["surfaceUpdate"]["components"]))

ua = user_action("approve_booking", "approve", "ok", {"flight": chosen["id"]})
audit("user_action", surface="approve", role="customer", action=ua["userAction"]["name"],
      context=ua["userAction"]["context"])

allowed, reason = authorize("book_flight", ua["userAction"]["context"], "customer", fare_int)
audit("authorization", action="book_flight", role="customer", allowed=allowed, reason=reason)

if allowed:
    book_flight(chosen["id"])
    audit("tool_executed", tool="book_flight", flight=chosen["id"], result="confirmed")
    conf = CAPTURE.payloads[-1]["messages"]
    audit("ui_emitted", surface="confirm", role="customer", components=len(conf[0]["surfaceUpdate"]["components"]))


def print_audit(rows):
    cols = [("ts", 22), ("kind", 16), ("detail", 60)]
    line = lambda c: "  ".join(str(v)[:w].ljust(w) for v, w in c)
    print(line([(h, w) for h, w in cols]))
    print(line([("-" * w, w) for _, w in cols]))
    for r in rows:
        detail = {k: v for k, v in r.items() if k not in ("ts", "kind")}
        print(line([(r["ts"], 22), (r["kind"], 16), (json.dumps(detail), 60)]))


print_audit(AUDIT)

ts                      kind              detail                                                      
----------------------  ----------------  ------------------------------------------------------------
2026-06-09T05:52:51Z    ui_emitted        {"surface": "flights", "role": "customer", "components": 17}
2026-06-09T05:52:51Z    ui_emitted        {"surface": "approve", "role": "customer", "components": 7} 
2026-06-09T05:52:51Z    user_action       {"surface": "approve", "role": "customer", "action": "approv
2026-06-09T05:52:51Z    authorization     {"action": "book_flight", "role": "customer", "allowed": tru
2026-06-09T05:52:51Z    tool_executed     {"tool": "book_flight", "flight": "JX511", "result": "confir
2026-06-09T05:52:51Z    ui_emitted        {"surface": "confirm", "role": "customer", "components": 3} 


## 8. A Streamlit admin view — SHOW (for the operator)

The record layer needs a lens. This writes the audit log and a small Streamlit app that shows the timeline, a payload inspector, and a render of what each surface looked like. On AWS this is where **AgentCore Observability** (CloudWatch + OpenTelemetry) plugs in. Notebook 2 covers that.

Streamlit runs as its own process, not inside Jupyter, so the cell writes the files and prints the run command.

In [13]:
# persist the run for the admin app
with open("audit_log.json", "w") as f:
    json.dump(AUDIT, f, indent=2)
with open("captured_surfaces.json", "w") as f:
    json.dump(CAPTURE.payloads, f, indent=2)
print("wrote audit_log.json and captured_surfaces.json")

wrote audit_log.json and captured_surfaces.json


In [14]:
admin_app = r'''
import json
import streamlit as st

st.set_page_config(page_title="TravelMind A2UI Admin", layout="wide")
st.title("TravelMind A2UI Admin")

audit = json.load(open("audit_log.json"))
surfaces = json.load(open("captured_surfaces.json"))

CONTAINER = {"Column", "Row"}


def resolve(b):
    if not isinstance(b, dict):
        return str(b)
    return b.get("literalString", b.get("path", ""))


def render_lines(messages):
    comps, root = {}, None
    for m in messages:
        if "surfaceUpdate" in m:
            for c in m["surfaceUpdate"]["components"]:
                comps[c["id"]] = c["component"]
        elif "beginRendering" in m:
            root = m["beginRendering"]["root"]
    out = []

    def walk(cid, d):
        node = comps.get(cid)
        if not node:
            return
        t = list(node)[0]
        body = node[t]
        pad = "  " * d
        if t == "Text":
            out.append(pad + resolve(body.get("text", {})))
        elif t == "Button":
            child = body.get("child")
            lbl = resolve(comps.get(child, {}).get("Text", {}).get("text", {})) if child in comps else child
            out.append(pad + f"[ {lbl} ]  -> {body['action']['name']}")
        elif t == "MetricCard":
            out.append(pad + f"+-- {resolve(body['label'])}: {resolve(body['value'])} --+")
        elif t in CONTAINER:
            out.append(pad + t + ":")
            for ch in body.get("children", {}).get("explicitList", []):
                walk(ch, d + 1)

    if root:
        walk(root, 0)
    return "\n".join(out)


c1, c2, c3, c4 = st.columns(4)
c1.metric("Surfaces emitted", sum(1 for a in audit if a["kind"] == "ui_emitted"))
c2.metric("User actions", sum(1 for a in audit if a["kind"] == "user_action"))
c3.metric("Authorizations", sum(1 for a in audit if a["kind"] == "authorization"))
c4.metric("Bookings", sum(1 for a in audit if a["kind"] == "tool_executed"))

st.subheader("Event timeline")
st.dataframe(audit, use_container_width=True)

st.subheader("Surface inspector")
labels = [f"{i}: {s['kind']} ({s['surfaceId']})" for i, s in enumerate(surfaces)]
if labels:
    pick = st.selectbox("Captured surface", range(len(labels)), format_func=lambda i: labels[i])
    left, right = st.columns(2)
    with left:
        st.caption("What the user saw")
        st.code(render_lines(surfaces[pick]["messages"]) or "(empty)")
    with right:
        st.caption("Raw A2UI payload")
        st.json(surfaces[pick]["messages"])
else:
    st.info("No surfaces captured yet.")
'''

with open("travelmind_admin.py", "w") as f:
    f.write(admin_app)
print("wrote travelmind_admin.py")
print("\nRun it in your terminal (not in Jupyter):")
print("  streamlit run travelmind_admin.py")

wrote travelmind_admin.py

Run it in your terminal (not in Jupyter):
  streamlit run travelmind_admin.py


## 9. Dos and don'ts

**Do**

| Do | Why |
|---|---|
| Validate every payload, fall back to text | The agent is an LLM; malformed UI will happen |
| Authorize the action server-side | The Approve button is convenience, not a control boundary |
| Log the payload and the userAction with timestamp and role | You must reproduce what the user saw and did |
| Keep the tool's return to the model short | The model needs "it worked", not the whole payload |
| Use the inference profile id | A bare model id throws `ValidationException` |

**Don't**

| Don't | Instead |
|---|---|
| Put business logic in the A2UI binding | v0.8 binding is 1:1 with no transformers; transform on the server |
| Trust `userAction.context` as-is | Re-resolve and re-check it against your own state |
| Render before `beginRendering` | Buffer messages, paint once, no flash of half-built UI |
| Reach for A2UI on a fixed, single-owner screen | Build it directly; A2UI earns its keep when UI is dynamic or crosses a trust boundary |
| Hard-code the booking after Approve | Run it only after the server gate clears |

Next: **Notebook 2** puts this same TravelMind agent on AgentCore, with Identity for roles, Policy for the gate, and Observability for the admin view.